# Try Genfleet rich input and output

Run from this SDK worktree with Python 3.12+: `python -m pip install -e ".[dev]"` and open this notebook in Jupyter. The main demo uses no API key or external service.

A turn is a list of typed parts (text, image, audio, video, file). The agent declares which file types it accepts; images reach a vision model as images, and every other file is described in one line, never sent as bytes.

## 1. Build parts with `part_for`

In [1]:
import base64, struct, zlib

from genfleet.sdk import AgentInput, TextPart, part_for


def tiny_png(width=2, height=2, rgb=(200, 40, 40)) -> bytes:
    def chunk(tag, body):
        crc = zlib.crc32(tag + body) & 0xFFFFFFFF
        return struct.pack(">I", len(body)) + tag + body + struct.pack(">I", crc)
    rows = b"".join(b"\x00" + bytes(rgb) * width for _ in range(height))
    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(rows)) + chunk(b"IEND", b""))

In [2]:
png = tiny_png()
image = part_for("image/png", data=png, name="red.png")
pdf = part_for("application/pdf", data=b"%PDF-1.4\n% a tiny stand-in report\n", name="report.pdf")
note = part_for("text/plain", data=b"Invoice 42 is due Friday.", name="note.txt")

In [3]:
from IPython.display import Image, display

display(Image(data=image.data, width=64))

In [4]:
turn = AgentInput(parts=[TextPart(text="What is "), TextPart(text="attached?"), image, pdf])
print([type(p).__name__ for p in turn.parts])
print("message derived from text parts:", repr(turn.message))
print(image.describe())

['TextPart', 'TextPart', 'ImagePart', 'FilePart']
message derived from text parts: 'What is attached?'
[Attached image: name="red.png", type=image/png, size=73 bytes]


## Use your own image

Set `IMAGE_PATH` to a PNG, JPEG, GIF or WebP on your machine. A file up to 1 MiB goes inline (`data`); a larger one, like most phone photos, goes by reference through a resolver, the way the platform sends files. The live cell at the end sends it to a real vision model. Skipped while `IMAGE_PATH` is `None`.

In [5]:
import mimetypes
from pathlib import Path

from genfleet.sdk import InMemoryFileResolver
from genfleet.sdk.files import read_part
from genfleet.sdk.parts import MAX_INLINE_BYTES

IMAGE_PATH = None  # e.g. "~/Desktop/photo.jpg"

my_image, my_resolver = None, InMemoryFileResolver()
if IMAGE_PATH:
    path = Path(IMAGE_PATH).expanduser()
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    raw = path.read_bytes()
    if len(raw) <= MAX_INLINE_BYTES:
        my_image = part_for(mime, data=raw, name=path.name)
    else:
        my_image = my_resolver.put(raw, mime_type=mime, name=path.name)
    print(type(my_image).__name__, my_image.mime_type, my_image.size, "bytes",
          "inline" if my_image.data else "by reference")
    shown = await read_part(my_image, max_bytes=20 * 1024 * 1024, metadata={}, resolver=my_resolver)
    display(Image(data=shown, width=300))
else:
    print("Set IMAGE_PATH to try your own image.")

Set IMAGE_PATH to try your own image.


## 2. An agent with a fake vision model: what the model receives

In [6]:
from collections.abc import AsyncIterator

from genfleet.sdk import Agent, AgentOutput


class DemoModel:
    def __init__(self):
        self.calls = []

    async def complete(self, messages, tools, stream=True) -> AsyncIterator[AgentOutput]:
        self.calls.append(messages)
        yield AgentOutput(content="Got it.", done=True)


def last_user(model):
    return [m for m in model.calls[-1] if m["role"] == "user"][-1]["content"]


def show(model):
    content = last_user(model)
    if isinstance(content, str):
        print("prompt is plain text:\n " + content.replace("\n", "\n "))
        return
    for block in content:
        if block["type"] == "text":
            print("text block:\n " + block["text"].replace("\n", "\n "))
        else:
            print("image block:", block["image_url"]["url"][:40] + "...")


async def ask(agent, **kwargs):
    return "".join([o.content or "" async for o in agent.run(AgentInput(**kwargs))])


model = DemoModel()
agent = Agent(
    role="You read invoices.",
    model={"model": "demo", "vision": True},
    model_client=model,
    accepts=["image/*", "application/pdf"],
)
await ask(agent, parts=[TextPart(text="Check these."), image, pdf])
show(model)
print("PDF bytes in prompt:", b"stand-in" in str(last_user(model)).encode())

text block:
 Check these.
 [Attached image: name="red.png", type=image/png, size=73 bytes]
 [Attached file: name="report.pdf", type=application/pdf, size=34 bytes]
image block: data:image/png;base64,iVBORw0KGgoAAAANSU...
PDF bytes in prompt: False


## 3. A file-only request (no text)

In [7]:
reply = await ask(agent, parts=[pdf])
print("reply:", reply, "| message:", repr(AgentInput(parts=[pdf]).message))
show(model)

reply: Got it. | message: ''
prompt is plain text:
 [Attached file: name="report.pdf", type=application/pdf, size=34 bytes]


## 4. An unsupported type is refused

In [8]:
from genfleet.sdk import UnsupportedInput

song = part_for("audio/mpeg", data=b"ID3....", name="song.mp3")
try:
    await ask(agent, parts=[TextPart(text="Transcribe"), song])
except UnsupportedInput as exc:
    print("UnsupportedInput:", exc)
    print("unsupported:", exc.unsupported, "accepted:", exc.accepted)

UnsupportedInput: Unsupported input type: audio/mpeg. This agent accepts: text/plain, image/*, application/pdf.
unsupported: ['audio/mpeg'] accepted: ['text/plain', 'image/*', 'application/pdf']


## 5. References with `InMemoryFileResolver`: good, expired, deduped and capped

In [9]:
from genfleet.sdk import InMemoryFileResolver


class CountingResolver(InMemoryFileResolver):
    reads = 0

    async def read(self, uri, **kwargs):
        self.reads += 1
        return await super().read(uri, **kwargs)


now = [0.0]  # a fake clock, so expiry needs no sleeping
resolver = CountingResolver(clock=lambda: now[0])
good = resolver.put(png, mime_type="image/png", name="good.png")
short = resolver.put(tiny_png(rgb=(0, 0, 255)), mime_type="image/png", name="short.png", ttl=60)
ref_model = DemoModel()
ref_agent = Agent(role="Look.", model={"model": "demo", "vision": True}, model_client=ref_model,
                  accepts=["image/*"], files=resolver, max_images_per_turn=2)

print("good ref:", good.uri[:14] + "...")
await ask(ref_agent, parts=[good]); show(ref_model)
now[0] = 120.0  # past the ttl
await ask(ref_agent, parts=[short]); print(); show(ref_model)

good ref: mem://7d437031...
text block:
 [Attached image: name="good.png", type=image/png, size=73 bytes]
image block: data:image/png;base64,iVBORw0KGgoAAAANSU...

prompt is plain text:
 [Attached image: name="short.png", type=image/png, size=72 bytes] (unavailable: expired)


In [10]:
resolver.reads = 0
await ask(ref_agent, parts=[good, good, good, good]); show(ref_model)
print("resolver reads for four copies:", resolver.reads)

resolver.reads = 0
distinct = [resolver.put(tiny_png(rgb=(i, i, i)), mime_type="image/png", name=f"g{i}.png") for i in (10, 20, 30)]
await ask(ref_agent, parts=distinct); print(); show(ref_model)
print("resolver reads with a cap of 2 images:", resolver.reads)

text block:
 [Attached image: name="good.png", type=image/png, size=73 bytes]
 [Attached image: name="good.png", type=image/png, size=73 bytes] (repeated; shown once)
 [Attached image: name="good.png", type=image/png, size=73 bytes] (repeated; shown once)
 [Attached image: name="good.png", type=image/png, size=73 bytes] (repeated; shown once)
image block: data:image/png;base64,iVBORw0KGgoAAAANSU...
resolver reads for four copies: 1

text block:
 [Attached image: name="g10.png", type=image/png, size=71 bytes]
 [Attached image: name="g20.png", type=image/png, size=71 bytes]
 [Attached image: name="g30.png", type=image/png, size=71 bytes] (not shown: image limit for this turn reached)
image block: data:image/png;base64,iVBORw0KGgoAAAANSU...
image block: data:image/png;base64,iVBORw0KGgoAAAANSU...
resolver reads with a cap of 2 images: 2


## 6. An agent returning a file part in `AgentOutput.parts`

In [11]:
class ReportAgent:
    accepted_input_types = ()
    output_types = ("text/csv",)

    async def run(self, input: AgentInput) -> AsyncIterator[AgentOutput]:
        csv = part_for("text/csv", data=b"item,total\ninvoice 42,120\n", name="totals.csv")
        yield AgentOutput(parts=[TextPart(text="Here are the totals."), csv], done=True)


async for out in ReportAgent().run(AgentInput(message="totals?")):
    print("content:", out.content)
    print("parts:", [(type(p).__name__, getattr(p, "name", None)) for p in out.parts])

content: Here are the totals.
parts: [('TextPart', None), ('FilePart', 'totals.csv')]


## 7. Serve it: A2A `SendMessage` with a file part, in process

In [12]:
from fastapi.testclient import TestClient

from genfleet.sdk.serve import create_app

served = Agent(role="You read invoices.", model={"model": "demo", "vision": True},
               model_client=DemoModel(), accepts=["image/*", "application/pdf"])
client = TestClient(create_app(served))

card = client.get("/.well-known/agent-card.json").json()
print("defaultInputModes:", card["defaultInputModes"])


def send(*parts):
    body = {"jsonrpc": "2.0", "id": 1, "method": "SendMessage",
            "params": {"message": {"messageId": "m1", "role": "ROLE_USER", "parts": list(parts)}}}
    return client.post("/", json=body).json()


ok = send({"text": "What is this?"},
          {"raw": base64.b64encode(png).decode(), "mediaType": "image/png", "filename": "red.png"})
print("reply:", ok["result"]["task"]["artifacts"][0]["parts"])

defaultInputModes: ['text/plain', 'image/*', 'application/pdf']
reply: [{'text': 'Got it.'}]


/Users/mmedhat/launchpad/genfleet/sdk-wt-rich-io-nb/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


In [13]:
bad = send({"text": "Transcribe"}, {"raw": base64.b64encode(b"ID3").decode(), "mediaType": "audio/mpeg"})
print(bad["error"]["code"], bad["error"]["message"])

-32005 Unsupported input type: audio/mpeg. This agent accepts: text/plain, image/*, application/pdf.


## Optional: a live vision model

Set `OPENROUTER_API_KEY` in your environment and install the SDK's `openai` extra. The default id is a free vision-capable model. A paid model is only used if you also set `RUN_PAID = True`. Skipped without a key. It describes your own image when `IMAGE_PATH` is set, otherwise a generated red square.

In [14]:
import os

RUN_PAID = False  # True switches to a paid model; leave False for the free one
FREE_VISION = "openrouter/google/gemma-4-31b-it:free"
PAID_VISION = "openrouter/google/gemini-2.0-flash-001"  # paid

openrouter_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_key:
    live = Agent(
        role="Describe the attached image in one short sentence.",
        model={"model": PAID_VISION if RUN_PAID else FREE_VISION, "api_key": openrouter_key, "vision": True},
        accepts=["image/*"],
        files=my_resolver,
    )
    if my_image is not None:
        print(await ask(live, parts=[TextPart(text="Describe this image."), my_image]))
    else:
        print(await ask(live, parts=[TextPart(text="What colour is this image?"), part_for("image/png", data=tiny_png(32, 32))]))
else:
    print("Set OPENROUTER_API_KEY to run the live cell.")

Set OPENROUTER_API_KEY to run the live cell.
